# Fine-tune ModernBERT into a System 1 decision model

A consumer complaint comes in. Which of 9 financial products is it about? We teach **ModernBERT-base**, a 149M
encoder model (a modern BERT), to answer in **one forward pass**: the complaint in, 9 probabilities out, no text
written. That's a System 1 decision. Same data and the same 826 test complaints as lesson 3, so we can compare it with
Laya and with lesson 3's mini-clef. Hugging Face `Trainer`, full fine-tune, on the Mac's GPU (MPS).

In [ ]:
import time
from collections import Counter

import torch
from datasets import load_dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, pipeline)

from router import ROUTES, score, scoreboard

## 0. Check the machine

The same **MacBook Pro** as every lesson. PyTorch reaches the Mac's GPU through **MPS**.

In [ ]:
!system_profiler SPHardwareDataType | grep -E "Model Name|Chip|Memory"

In [ ]:
torch.backends.mps.is_available()

## 1. The data: consumer complaints, 9 products

[`kgptalkie/cfpb-complaint-routing`](https://huggingface.co/datasets/kgptalkie/cfpb-complaint-routing) (CC0): short
complaints from the US CFPB's public complaint database, each labelled with its product. The 826 test complaints are
the ones Laya was scored on.

In [ ]:
data = load_dataset("kgptalkie/cfpb-complaint-routing")
train, val, test = data["train"], data["validation"], data["test"].to_list()
data

In [ ]:
ex = next(r for r in test if r["id"] == "cfpb-4356244")
print(ex["complaint"])
ex["product"]

In [ ]:
Counter(train["product"])

## 2. ModernBERT

[`answerdotai/ModernBERT-base`](https://huggingface.co/answerdotai/ModernBERT-base) (Apache-2.0, Answer.AI and LightOn):
an **encoder**. A chat model like Qwen reads left to right and writes the next word. An encoder reads the whole text at
once, every word looking both ways, and turns it into vectors. It never writes. The tokenizer splits the complaint into
tokens:

In [ ]:
MODEL = "answerdotai/ModernBERT-base"
tok = AutoTokenizer.from_pretrained(MODEL)
ids = tok(ex["complaint"]).input_ids
print(len(ids), "tokens")
print(tok.convert_ids_to_tokens(ids)[:16])

## 3. A decision head

`AutoModelForSequenceClassification` puts a decision **head** on top: it averages the token vectors into one vector
for the whole complaint, passes it through a small layer, and a final layer turns those 768 numbers into **9 scores**,
one per product. A softmax makes them probabilities. That final layer is new (the load report lists it as MISSING),
so it starts random and has to learn.

In [ ]:
torch.manual_seed(0)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(ROUTES), id2label=dict(enumerate(ROUTES)), label2id={r: i for i, r in enumerate(ROUTES)})
print(model.classifier)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

Untrained, it's a guess. On all 826 test complaints, 16 at a time:

In [ ]:
router = pipeline("text-classification", model=model, tokenizer=tok, device="mps", top_k=3)
print(router(ex["complaint"]))
before, _ = score(router, test)
f"{before:.1%}"

## 4. Training examples

Each example is the complaint's tokens and its product number (0–8), cut at 256 tokens (the longest complaint has 227).

In [ ]:
def encode(ds):
    return ds.map(lambda b: {**tok(b["complaint"], truncation=True, max_length=256),
                             "label": [ROUTES.index(p) for p in b["product"]]},
                  batched=True, remove_columns=ds.column_names)

train_ds, val_ds = encode(train), encode(val)
train_ds

## 5. Train with the Hugging Face Trainer

A **full fine-tune**: every one of the 149M weights trains, no LoRA needed at this size. 3 passes over the 2,700
training complaints, 16 at a time, with the accuracy on the 270 validation complaints after each pass. Batches are
padded to a multiple of 64 tokens: MPS caches its GPU work per tensor shape, so fewer shapes means less memory.

In [ ]:
def accuracy(p):
    return {"accuracy": (p.predictions.argmax(-1) == p.label_ids).mean()}

args = TrainingArguments(output_dir="checkpoints", num_train_epochs=3, per_device_train_batch_size=16,
                         learning_rate=5e-5, lr_scheduler_type="linear", warmup_steps=0.1, weight_decay=0.01,
                         eval_strategy="epoch", logging_steps=50, save_strategy="no", bf16=True,
                         report_to="none", seed=0)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok, pad_to_multiple_of=64), compute_metrics=accuracy)

In [ ]:
torch.mps.empty_cache()
start = time.time()
trainer.train()
train_secs = time.time() - start
gpu_gb = torch.mps.driver_allocated_memory() / 1e9
print(f"{train_secs / 60:.1f} min, GPU memory {gpu_gb:.1f} GB")

## 6. Save it

One normal Hugging Face model folder: the weights, the head, the 9 route names and the tokenizer.

In [ ]:
trainer.save_model("modernbert-complaint-router")
tok.save_pretrained("modernbert-complaint-router")

## 7. Use it: one line with `pipeline`

The same car-loan complaint, then a new one.

In [ ]:
router = pipeline("text-classification", model="modernbert-complaint-router", device="mps", top_k=3)
router(ex["complaint"])

In [ ]:
router("I sent $400 with Zelle to pay my landlord, but it went to the wrong person and nobody will reverse it.")

## 8. Before vs after

The same 826 test complaints: 16 at a time, then one at a time, the way a live app would ask.

In [ ]:
after, ms = score(router, test)
_, ms_one = score(router, test, batch_size=1)
print(f"{after:.1%} right · {ms:.1f} ms per complaint (16 at a time) · {ms_one:.1f} ms (one at a time)")

In [ ]:
scoreboard(before, after, ms, train_secs, gpu_gb, trainer.state.global_step, args.per_device_train_batch_size)

## 9. Share it on Hugging Face

The model folder and its model card go to a (private) repo on your account. Log in once with `hf auth login`.

In [ ]:
from huggingface_hub import HfApi

REPO = "kgptalkie/modernbert-complaint-router"
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path="modernbert-complaint-router", repo_id=REPO, commit_message="ModernBERT-base + 9-route head, 3 epochs")
api.upload_file(path_or_fileobj="MODEL_CARD.md", path_in_repo="README.md", repo_id=REPO)
print(f"https://huggingface.co/{REPO}")

## That's the series

Ten lessons, ten working models, all trained on a Mac. Thanks for building them with me.